# Example 2: Simple 2D cell signaling model

We model a reaction between the cell interior and cell membrane in a 2D geometry:
- Cyto - 2D cell "volume"
- PM - 1D cell boundary (represents plasma membrane)

Model from [Rangamani et al, 2013, Cell](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3874130/). A cytosolic species, "A", reacts with a species on the PM, "B", to form a new species on the PM, "X". The resulting PDE and boundary condition for species A are as follows:

$$
\frac{\partial{C_A}}{\partial{t}} = D_A \nabla ^2 C_A \quad \text{in} \; \Omega_{Cyto}\\
\text{B.C. for A:} \quad D_A (\textbf{n} \cdot \nabla C_A)  = -k_{on} C_A N_X + k_{off} N_B \quad \text{on} \; \Gamma_{PM}
$$

Similarly, the PDEs for X and B are given by:
$$
\frac{\partial{N_X}}{\partial{t}} = D_X \nabla ^2 N_X - k_{on} C_A N_X + k_{off} N_B \quad \text{on} \; \Gamma_{PM}\\
\frac{\partial{N_B}}{\partial{t}} = D_B \nabla ^2 N_B + k_{on} C_A N_X - k_{off} N_B \quad \text{on} \; \Gamma_{PM}
$$

In [ ]:
from matplotlib import pyplot as plt
import matplotlib.image as mpimg
img_A = mpimg.imread('axb-diagram.png')
plt.imshow(img_A)
plt.axis('off')

Imports and logger initialization:

In [ ]:
import dolfin as d
import sympy as sym
import numpy as np
import pathlib
import logging
import gmsh  # must be imported before pyvista if dolfin is imported first

from smart import config, common, mesh, model, mesh_tools, visualization
from smart.units import unit
from smart.model_assembly import (
    Compartment,
    Parameter,
    Reaction,
    Species,
    SpeciesContainer,
    ParameterContainer,
    CompartmentContainer,
    ReactionContainer,
    Form,
)
from matplotlib import pyplot as plt
import matplotlib.image as mpimg

logger = logging.getLogger("smart")
logger.setLevel(logging.INFO)

First, we define the various units for use in the model.

In [ ]:
um = unit.um
molecule = unit.molecule
sec = unit.sec
dimensionless = unit.dimensionless
D_unit = um**2 / sec
surf_unit = molecule / um**2
flux_unit = molecule / (um * sec)
edge_unit = molecule / um

Next we generate the model by assembling the compartment, species, parameter, and reaction containers (see Example 1 or API documentation for more details).

In [ ]:
# =============================================================================================
# Compartments
# =============================================================================================
# name, topological dimensionality, length scale units, marker value
# Cyto = Compartment("Cyto", 2, um, 1, vel=("-0.1*y*(1 - (x**2+y**2))","0.1*x*(1 - (x**2+y**2))","0"))
# Cyto = Compartment("Cyto", 2, um, 1, deform=("-0.1*y*t*(1 - (x**2+y**2))","0.1*x*t*(1 - (x**2+y**2))","0"), manual_update=True)
Cyto = Compartment("Cyto", 2, um, 1, deform=("((1+sign(t-10))/2)*0.005*(t-10)*x/(1-0.005*(t-10))",
                                             "-((1+sign(t-10))/2)*0.005*(t-10)*y","0")) # dilation
PM = Compartment("PM", 1, um, 3, deform=("((1+sign(t-10))/2)*0.005*(t-10)*x/(1-0.005*(t-10))",
                                         "-((1+sign(t-10))/2)*0.005*(t-10)*y","0"))
cc = CompartmentContainer()
cc.add([Cyto, PM])

# =============================================================================================
# Species
# =============================================================================================
# name, initial concentration, concentration units, diffusion, diffusion units, compartment
# A = Species("A", "10*exp(-(x**2+(y-0.8)**2)/0.2**2)", surf_unit, 0.001, D_unit, "Cyto")
A = Species("A", 1.0, surf_unit, 0.001, D_unit, "Cyto")
X = Species("X", 1.0, edge_unit, 0.001, D_unit, "PM")
B = Species("B", 0.0, edge_unit, 0.001, D_unit, "PM")
sc = SpeciesContainer()
sc.add([A, X, B])

# =============================================================================================
# Parameters and Reactions
# =============================================================================================

# Reaction of A and X to make B (Cyto-PM reaction)
kon = Parameter("kon", 1.0, 1/(surf_unit*sec))
koff = Parameter("koff", 1.0, 1/sec)
r1 = Reaction("r1", ["A", "X"], ["B"],
              param_map={"on": "kon", "off": "koff"},
              species_map={"A": "A", "X": "X", "B": "B"})

pc = ParameterContainer()
pc.add([kon, koff])
rc = ReactionContainer()
rc.add([r1])

Now we create a circular mesh (mesh built using gmsh in `smart.mesh_tools`), along with marker functions `mf2` and `mf1`.

In [ ]:
# Create mesh
h_ellipse = 0.1
xrad = 1.0
yrad = 1.0
surf_tag = 1
edge_tag = 3
ellipse_mesh, mf1, mf2 = mesh_tools.create_ellipses(2*xrad, 0.5*yrad, hEdge=h_ellipse,
                                                    outer_tag=surf_tag, outer_marker=edge_tag)
visualization.plot_dolfin_mesh(ellipse_mesh, mf2, view_xy=True)

Write mesh and meshfunctions to file, then create `mesh.ParentMesh` object.

In [ ]:
mesh_folder = pathlib.Path("ellipse_mesh_AR1")
mesh_folder.mkdir(exist_ok=True)
mesh_file = mesh_folder / "ellipse_mesh.h5"
mesh_tools.write_mesh(ellipse_mesh, mf1, mf2, mesh_file)

parent_mesh = mesh.ParentMesh(
    mesh_filename=str(mesh_file),
    mesh_filetype="hdf5",
    name="parent_mesh",
)

Initialize model and solvers.

In [ ]:
config_cur = config.Config()
config_cur.solver.update(
    {
        "final_t": 130.0,
        "initial_dt": 1.0,
        "time_precision": 6,
    }
)

model_cur = model.Model(pc, sc, cc, rc, config_cur, parent_mesh)
model_cur.initialize()

Save model information to .pkl file and write initial conditions to file.

In [ ]:
model_cur.to_pickle('model_cur.pkl')
results = dict()
result_folder = pathlib.Path("resultsEllipse_testShapeChange")
result_folder.mkdir(exist_ok=True)
for species_name, species in model_cur.sc.items:
    results[species_name] = d.XDMFFile(
        model_cur.mpi_comm_world, str(result_folder / f"{species_name}.xdmf")
    )
    results[species_name].parameters["flush_output"] = True
    results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)

# save deformation or velocity field as well
if cc["Cyto"].vel_logic:
    velfile = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"vel.xdmf"))
    velfile.parameters["flush_output"] = True
    velfile.write(sc["A"].compartment.vel_func, model_cur.t)
elif cc["Cyto"].deform_logic:
    u1file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"u1var.xdmf"))
    u1file.parameters["flush_output"] = True
    u1file.write(cc["Cyto"].deform_func, model_cur.t)
    u2file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"u2var.xdmf"))
    u2file.parameters["flush_output"] = True
    u2file.write(cc["PM"].deform_func, model_cur.t)
    # v1file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"v1.xdmf"))
    # v1file.parameters["flush_output"] = True
    # v2file = d.XDMFFile(model_cur.mpi_comm_world, str(result_folder / f"v2.xdmf"))
    # v2file.parameters["flush_output"] = True

Solve the system until `model_cur.t > model_cur.final_t`.

In [ ]:
tvec = [0]
dx_Cyto = d.Measure("dx", domain=model_cur.cc['Cyto'].dolfin_mesh)
dx_PM = d.Measure("dx", domain=model_cur.cc['PM'].dolfin_mesh)
volume = d.assemble_mixed(1.0*dx_Cyto)
PM_SA = d.assemble_mixed(1.0*dx_PM)
avg_A = [d.assemble_mixed(A.u['u']*dx_Cyto) / volume]
avg_X = [d.assemble_mixed(X.u["u"]*dx_PM) / PM_SA]
avg_B = [d.assemble_mixed(B.u["u"]*dx_PM) / PM_SA]
# Set loglevel to warning in order not to pollute notebook output
logger.setLevel(logging.WARNING)
x = d.SpatialCoordinate(sc["A"].compartment.dolfin_mesh)
# if cc["Cyto"].deform_logic:
#     udef = cc["Cyto"].deform_func
#     udef_local = udef.copy()
#     V_vector = udef.function_space()
#     vel_expr = d.Expression(("-0.1*x[1]*(1 - (pow(x[0],2)+pow(x[1],2)))",
#                              "0.1*x[0]*(1 - (pow(x[0],2)+pow(x[1],2)))","0"), degree=1)
#     vel = d.interpolate(vel_expr, V_vector)
    # v1file.write(vel, model_cur.t)
    # vel_approx = d.project((udef - cc["Cyto"].deform_prev) / model_cur.dT, V_vector)
    # v2file.write(vel_approx, model_cur.t)

while True:
    # Solve the system
    model_cur.monolithic_solve()
    # Save results for post processing
    for species_name, species in model_cur.sc.items:
        results[species_name].write(model_cur.sc[species_name].u["u"], model_cur.t)
    avg_A.append(d.assemble_mixed(A.u['u']*dx_Cyto) / volume)
    avg_X.append(d.assemble_mixed(X.u["u"]*dx_PM) / PM_SA)
    avg_B.append(d.assemble_mixed(B.u["u"]*dx_PM) / PM_SA)
    tvec.append(model_cur.t)
    # save deformation or velocity
    if cc["Cyto"].vel_logic:
        velfile.write(sc["A"].compartment.vel_func, model_cur.t)
    elif cc["Cyto"].deform_logic:
    #     cc["Cyto"].deform_prev.assign(udef_local)
    #     udef.assign(udef_local + vel*d.Constant(model_cur.tvec[-1]-model_cur.tvec[-2]))
        u1file.write(cc["Cyto"].deform_func, model_cur.t)
        u2file.write(cc["PM"].deform_func, model_cur.t)
        # multCur = d.project(model_cur.fc["r1 [X (f)]"].integral_factor,X.V)
        # if np.any(np.isnan(multCur.vector()[:])):
        # udef = cc["Cyto"].deform_func
        # Fcur = d.Identity(3) + d.grad(udef)
        # Jcur = d.det(Fcur)
        # Nexpr = d.Expression(("x[0]/R", "x[1]/R", "0.0"), degree=1, R=1)
        # test_factor = Jcur*d.sqrt(d.inner(d.dot(Nexpr,d.inv(Fcur)),d.dot(Nexpr,d.inv(Fcur))))
        # Vcur = d.VectorFunctionSpace(X.compartment.dolfin_mesh, "P", 1)
        # N = d.interpolate(Nexpr, Vcur)
        # N = self.surface.normals
        
        # for name, flux in model_cur.fc.items:
        #     if "r1" in name:
        #         Vcur = flux.integral_factor.function_space()
        #         not_assigned = True
        #         while not_assigned:
        #             try:
        #                 flux.integral_factor.assign(d.project(test_factor, Vcur))
        #                 not_assigned = False
        #             except:
        #                 print('Try again!')
        # multfile.write(model_cur.fc["r1 [X (f)]"].integral_factor, model_cur.t)
    #     udef_local = udef.copy()
    #     # compute new velocities
    #     xcur = x[0] + udef[0]
    #     ycur = x[1] + udef[1]
    #     vcur_x = d.project(-0.1*ycur*(1 - (xcur**2+ycur**2)), V_vector.sub(0).collapse())
    #     vcur_y = d.project(0.1*xcur*(1 - (xcur**2+ycur**2)), V_vector.sub(1).collapse())
    #     vel.vector()[V_vector.sub(0).dofmap().dofs()] = vcur_x.vector()
    #     vel.vector()[V_vector.sub(1).dofmap().dofs()] = vcur_y.vector()
        # vel.vector().apply("insert")
        # v1file.write(vel, model_cur.t)
        # vel_approx.assign(d.project((udef - cc["Cyto"].deform_prev) / model_cur.dT, V_vector))
        # v2file.write(vel_approx, model_cur.t)

    print(f"Done with t={model_cur.t}")
    # End if we've passed the final time
    if model_cur.t >= model_cur.final_t:
        break

Now we plot the concentration of A in the cell over time and compare this to analytical predictions for a high value of the diffusion coefficient. As $D_A \rightarrow \infty$, the steady state concentration of A will be given by the positive root of the following polynomial:

$$
-k_{on} c_A ^2 - \left( k_{on} c_{X} (t=0) \frac{SA_{PM}}{vol_{cyto}} + k_{off} - k_{on} c_A (t=0)   \right) c_A + k_{off} c_A(t=0)
$$

Note that in this 2D case, $SA_{PM}$ is the perimeter of the ellipse and $vol_{cyto}$ is the area of the ellipse. These can be thought of as a surface area and volume if we extrude the 2D shape by some characteristic thickness.

In [ ]:
plt.plot(tvec, avg_A, label='SMART simulation')
plt.xlabel('Time (s)')
plt.ylabel('A concentration $\mathrm{(molecules/μm^2)}$')
SA_vol = 4/(xrad + yrad)
root_vals = np.roots([-kon.value,
                      -kon.value*avg_X[0]*SA_vol - koff.value + kon.value*avg_A[0],
                      koff.value*avg_A[0]])
ss_pred = root_vals[root_vals > 0]
plt.plot(tvec, np.ones(len(avg_A))*ss_pred, '--', label='Steady-state analytical prediction')
plt.legend()
# from cowpy import cow
# if True:#percent_error_analytical > 0.1:
#     stegy = cow.Stegosaurus(thoughts=True)
#     stegy_msg = stegy.milk("Failed test: Example 2 results deviate from the analytical prediction")
#     print(stegy_msg)

Plot A concentration in the cell at the final time point.

In [ ]:
visualization.plot(model_cur.sc["A"].u["u"], view_xy=True)